# RUL Prediction: Hybrid Transformer Models
## LSTM-Transformer, GRU-Transformer, CNN-Transformer Architectures
**Owen's Implementation - Team Deep Learning Project**

# Hyperparameter Tuning (FD001)
- Models: LSTM-Transformer, GRU-Transformer, CNN-Transformer.
- Datasets: FE - Manual (Drop s14), FE - Manual (Drop s14, s11).
- Tune: sequence length and learning rate.
- Metric: RMSE for train, val, test (select best by val RMSE).
- Save best model per seed into: owen/models/hyper.

In [25]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

SEEDS = [1234, 42, 999]
BATCH_SIZE = 64
TRAIN_TEST_SPLIT = 0.8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

Using device: mps


In [26]:
%cd Tearm_8/50.039_DL/DL_Project/
cwd = os.getcwd()
print("Current working directory:", cwd)

[Errno 2] No such file or directory: 'Tearm_8/50.039_DL/DL_Project/'
/Users/owensengdalavong/Desktop/Tearm_8/50.039_DL/DL_Project
Current working directory: /Users/owensengdalavong/Desktop/Tearm_8/50.039_DL/DL_Project


In [27]:
# Two best datasets for tuning
train_df_low_variance_1_125 = pd.read_csv(
    'sutd_50039_deep_learning/data/processed-nasa-data/feature_engineering_2/low_variance_1/train_fd001_low_variance_1_125.csv'
 )
test_df_low_variance_1_125 = pd.read_csv(
    'sutd_50039_deep_learning/data/processed-nasa-data/feature_engineering_2/low_variance_1/test_fd001_low_variance_1_125.csv'
 )

DATASETS_CONFIG = {
    'FE Low Variance 125': (train_df_low_variance_1_125, test_df_low_variance_1_125),
}

In [28]:
class RULDataset(Dataset):
    def __init__(self, df, sequence_length=30, feature_cols=None):
        self.sequence_length = sequence_length
        if feature_cols is None:
            self.feature_cols = [col for col in df.columns if col[0] == 's' and col[1:].isdigit()]
        else:
            self.feature_cols = feature_cols

        self.sequences = []
        self.labels = []
        for engine_id in df['id'].unique():
            engine_data = df[df['id'] == engine_id].sort_values('cycle').reset_index(drop=True)
            features = engine_data[self.feature_cols].values
            rul = engine_data['RUL'].values
            for i in range(len(engine_data) - sequence_length):
                self.sequences.append(features[i:i + sequence_length])
                self.labels.append(rul[i + sequence_length])

        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx]), torch.tensor(self.labels[idx])

def build_loaders_by_seed(sequence_length):
    base_datasets = {}
    test_datasets = {}
    for dataset_name, (train_df, test_df) in DATASETS_CONFIG.items():
        train_dataset = RULDataset(train_df, sequence_length)
        test_dataset = RULDataset(test_df, sequence_length, feature_cols=train_dataset.feature_cols)
        base_datasets[dataset_name] = train_dataset
        test_datasets[dataset_name] = test_dataset

    loaders_by_seed = {}
    for seed in SEEDS:
        seed_everything(seed)
        loaders_by_seed[seed] = {}
        for dataset_name in base_datasets.keys():
            train_dataset = base_datasets[dataset_name]
            test_dataset = test_datasets[dataset_name]
            train_size = int(TRAIN_TEST_SPLIT * len(train_dataset))
            val_size = len(train_dataset) - train_size
            train_split, val_split = random_split(
                train_dataset,
                [train_size, val_size],
                generator=torch.Generator().manual_seed(seed)
            )
            train_loader = DataLoader(
                train_split,
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=0,
                worker_init_fn=seed_worker
            )
            val_loader = DataLoader(
                val_split,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=0,
                worker_init_fn=seed_worker
            )
            test_loader = DataLoader(
                test_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=0,
                worker_init_fn=seed_worker
            )
            loaders_by_seed[seed][dataset_name] = {
                'train': train_loader,
                'val': val_loader,
                'test': test_loader,
                'num_features': len(train_dataset.feature_cols)
            }
    return loaders_by_seed

In [29]:
class LSTMTransformer(nn.Module):
    def __init__(self, num_features, lstm_hidden=64, num_lstm_layers=2, d_model=64, nhead=4, num_transformer_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=lstm_hidden,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )
        self.projection = nn.Linear(lstm_hidden, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        transformer_in = self.projection(lstm_out)
        transformer_out = self.transformer(transformer_in)
        last_output = transformer_out[:, -1, :]
        return self.fc(last_output).squeeze(-1)

class GRUTransformer(nn.Module):
    def __init__(self, num_features, gru_hidden=64, num_gru_layers=2, d_model=64, nhead=4, num_transformer_layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_features,
            hidden_size=gru_hidden,
            num_layers=num_gru_layers,
            batch_first=True,
            dropout=dropout if num_gru_layers > 1 else 0
        )
        self.projection = nn.Linear(gru_hidden, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        gru_out, _ = self.gru(x)
        transformer_in = self.projection(gru_out)
        transformer_out = self.transformer(transformer_in)
        last_output = transformer_out[:, -1, :]
        return self.fc(last_output).squeeze(-1)

class CNNTransformer(nn.Module):
    def __init__(self, num_features, num_filters=32, kernel_size=3, d_model=64, nhead=4, num_transformer_layers=2, dropout=0.1):
        super().__init__()
        self.conv1d = nn.Sequential(
            nn.Conv1d(
                in_channels=num_features,
                out_channels=num_filters,
                kernel_size=kernel_size,
                padding=kernel_size // 2
            ),
            nn.ReLU(),
            nn.BatchNorm1d(num_filters),
            nn.Dropout(dropout)
        )
        self.projection = nn.Linear(num_filters, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        cnn_out = self.conv1d(x)
        cnn_out = cnn_out.transpose(1, 2)
        transformer_in = self.projection(cnn_out)
        transformer_out = self.transformer(transformer_in)
        last_output = transformer_out[:, -1, :]
        return self.fc(last_output).squeeze(-1)

In [35]:
def evaluate_rmse(model, data_loader, device):
    model.eval()
    total_sse = 0.0
    total_count = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            total_sse += torch.sum((preds - y_batch) ** 2).item()
            total_count += y_batch.numel()
    return np.sqrt(total_sse / max(total_count, 1))

def train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    device,
    lr,
    epochs,
    meta=None,
    remaining_trials=None
 ):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_test = float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = loss_fn(preds, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        test_rmse = evaluate_rmse(model, test_loader, device)
        best_test = min(best_test, test_rmse)
        if meta is not None:
            remaining = "" if remaining_trials is None else f" | remaining trials: {remaining_trials}"
            print(
                f"Epoch {epoch}/{epochs} | seed={meta['seed']} | {meta['dataset']} | {meta['model']}"
                f" | seq={meta['seq_len']} | lr={meta['lr']} | test_rmse={test_rmse:.4f}"
                f" | best_test={best_test:.4f}{remaining}"
            )
    train_rmse = evaluate_rmse(model, train_loader, device)
    val_rmse = evaluate_rmse(model, val_loader, device)
    return model, train_rmse, val_rmse

In [ ]:
SEQ_LENS = [30, 50, 100]
LRS = [1e-4, 5e-4, 1e-3]
D_MODELS = [64, 128]
EPOCHS = 5

MODEL_FACTORIES = {
    'LSTM-Transformer': lambda nf, d: LSTMTransformer(num_features=nf, lstm_hidden=64, num_lstm_layers=2, d_model=d, nhead=4, num_transformer_layers=2, dropout=0.1),
    'GRU-Transformer': lambda nf, d: GRUTransformer(num_features=nf, gru_hidden=64, num_gru_layers=2, d_model=d, nhead=4, num_transformer_layers=2, dropout=0.1),
    'CNN-Transformer': lambda nf, d: CNNTransformer(num_features=nf, num_filters=32, kernel_size=3, d_model=d, nhead=4, num_transformer_layers=2, dropout=0.1),
}

save_dir = Path('sutd_50039_deep_learning/owen/models/hyper')
save_dir.mkdir(parents=True, exist_ok=True)
log_dir = Path('sutd_50039_deep_learning/owen/logs')
log_dir.mkdir(parents=True, exist_ok=True)

results = []
total_trials = len(SEQ_LENS) * len(SEEDS) * len(DATASETS_CONFIG) * len(MODEL_FACTORIES) * len(LRS) * len(D_MODELS)
trial_idx = 0

for seq_len in SEQ_LENS:
    loaders_by_seed = build_loaders_by_seed(seq_len)
    for seed in SEEDS:
        seed_everything(seed)
        for dataset_name in DATASETS_CONFIG.keys():
            for model_name, make_model in MODEL_FACTORIES.items():
                best_val = float('inf')
                best_row = None
                best_state = None
                for d_model in D_MODELS:
                    for lr in LRS:
                        trial_idx += 1
                        loaders = loaders_by_seed[seed][dataset_name]
                        model = make_model(loaders['num_features'], d_model)
                        meta = {
                            'seed': seed,
                            'dataset': dataset_name,
                            'model': model_name,
                            'seq_len': seq_len,
                            'lr': lr
                        }
                        remaining = total_trials - trial_idx
                        model, train_rmse, val_rmse = train_model(
                            model,
                            loaders['train'],
                            loaders['val'],
                            loaders['test'],
                            DEVICE,
                            lr=lr,
                            epochs=EPOCHS,
                            meta=meta,
                            remaining_trials=remaining
                        )
                        test_rmse = evaluate_rmse(model, loaders['test'], DEVICE)
                        row = {
                            'Seed': seed,
                            'Dataset': dataset_name,
                            'Model': model_name,
                            'Seq Len': seq_len,
                            'LR': lr,
                            'D Model': d_model,
                            'Train RMSE': train_rmse,
                            'Val RMSE': val_rmse,
                            'Test RMSE': test_rmse
                        }
                        results.append(row)
                        if val_rmse < best_val:
                            best_val = val_rmse
                            best_row = row
                            best_state = model.state_dict()
                if best_state is not None:
                    safe_dataset = dataset_name.replace(' ', '_').replace('(', '').replace(')', '')
                    safe_model = model_name.replace('-', '_')
                    model_path = save_dir / f"{safe_model}_{safe_dataset}_seed{seed}_seq{best_row['Seq Len']}_d{best_row['D Model']}_lr{best_row['LR']}.pth"
                    torch.save(best_state, model_path)
                    print(f"Saved best: {model_path}")

results_df = pd.DataFrame(results)
results_df.sort_values(['Dataset', 'Model', 'Seed', 'Val RMSE'], inplace=True)

best_idx = results_df['Test RMSE'].idxmin()
best_row = results_df.loc[best_idx]
print(
    "Best overall | ",
    f"Dataset={best_row['Dataset']} | Model={best_row['Model']} | Seed={best_row['Seed']} | ",
    f"Seq={best_row['Seq Len']} | D Model={best_row['D Model']} | LR={best_row['LR']} | ",
    f"Train RMSE={best_row['Train RMSE']:.4f} | Val RMSE={best_row['Val RMSE']:.4f} | Test RMSE={best_row['Test RMSE']:.4f}"
 )

results_csv = log_dir / "hyper_results.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved results: {results_csv}")

results_df

Epoch 1/5 | seed=1234 | FE Low Variance 125 | LSTM-Transformer | seq=30 | lr=0.0001 | test_rmse=101.3399 | best_test=101.3399 | remaining trials: 8
Epoch 2/5 | seed=1234 | FE Low Variance 125 | LSTM-Transformer | seq=30 | lr=0.0001 | test_rmse=86.4361 | best_test=86.4361 | remaining trials: 8
Epoch 3/5 | seed=1234 | FE Low Variance 125 | LSTM-Transformer | seq=30 | lr=0.0001 | test_rmse=65.7516 | best_test=65.7516 | remaining trials: 8
Epoch 4/5 | seed=1234 | FE Low Variance 125 | LSTM-Transformer | seq=30 | lr=0.0001 | test_rmse=48.4158 | best_test=48.4158 | remaining trials: 8
Epoch 5/5 | seed=1234 | FE Low Variance 125 | LSTM-Transformer | seq=30 | lr=0.0001 | test_rmse=34.6091 | best_test=34.6091 | remaining trials: 8
Saved best: sutd_50039_deep_learning/owen/models/hyper/LSTM_Transformer_FE_Low_Variance_125_seed1234_seq30_lr0.0001.pth
Epoch 1/5 | seed=1234 | FE Low Variance 125 | GRU-Transformer | seq=30 | lr=0.0001 | test_rmse=99.0081 | best_test=99.0081 | remaining trials: 7
Epo

,Seed,Dataset,Model,Seq Len,LR,Train RMSE,Val RMSE,Test RMSE
5,42,FE Low Variance 125,CNN-Transformer,30,0.0001,18.858904,19.083391,21.597876
8,999,FE Low Variance 125,CNN-Transformer,30,0.0001,20.218271,20.226466,24.183553
2,1234,FE Low Variance 125,CNN-Transformer,30,0.0001,20.488302,20.510314,23.750163
4,42,FE Low Variance 125,GRU-Transformer,30,0.0001,21.222684,21.061894,25.212465
7,999,FE Low Variance 125,GRU-Transformer,30,0.0001,21.060042,21.047711,25.101994
1,1234,FE Low Variance 125,GRU-Transformer,30,0.0001,19.875096,19.806994,23.866760
3,42,FE Low Variance 125,LSTM-Transformer,30,0.0001,30.883445,30.434399,36.807045
6,999,FE Low Variance 125,LSTM-Transformer,30,0.0001,27.307039,27.524813,31.996995
0,1234,FE Low Variance 125,LSTM-Transformer,30,0.0001,32.127774,31.946319,34.609062
